# Epic 9.01 — Crack Geometry and Type Generation

This notebook demonstrates the crack geometry engine and the four
semiconductor-specific crack types.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from udm_epic9.models.crack_geometry import (
    generate_crack_path, generate_branching_crack,
    generate_crack_network, render_crack_mask,
)
from udm_epic9.models.crack_types import (
    die_crack, substrate_crack, mold_crack, delamination_crack,
)
from udm_epic9.rendering.usm_renderer import (
    render_crack_on_usm, generate_synthetic_usm_with_cracks,
)

## Crack Path Generation

The `generate_crack_path` function uses random midpoint displacement
(fractal subdivision) to create realistic rough crack paths.

In [ ]:
rng = np.random.default_rng(42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, roughness in zip(axes, [0.1, 0.3, 0.5]):
    path = generate_crack_path((10, 10), (240, 240), roughness=roughness, n_points=50, rng=rng)
    mask = render_crack_mask([path], 256, 256, width_range=(1, 3), rng=rng)
    ax.imshow(mask, cmap='gray')
    ax.set_title(f'Roughness = {roughness}')
    ax.axis('off')
plt.suptitle('Crack Path Roughness Comparison')
plt.tight_layout()
plt.show()

## Branching and Network Cracks

In [ ]:
rng = np.random.default_rng(123)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Branching crack
branch_paths = generate_branching_crack(
    origin=(128, 128), n_branches=4, max_depth=3,
    rng=rng, height=256, width=256,
)
branch_mask = render_crack_mask(branch_paths, 256, 256, rng=rng)
axes[0].imshow(branch_mask, cmap='gray')
axes[0].set_title(f'Branching ({len(branch_paths)} paths)')

# Network crack
net_paths = generate_crack_network(256, 256, n_cracks=6, rng=rng)
net_mask = render_crack_mask(net_paths, 256, 256, rng=rng)
axes[1].imshow(net_mask, cmap='gray')
axes[1].set_title(f'Network ({len(net_paths)} paths)')

for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

## Four Semiconductor Crack Types

Each type corresponds to a common failure mode in semiconductor packaging.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
generators = [
    ('Die Crack', die_crack),
    ('Substrate Crack', substrate_crack),
    ('Mold Crack', mold_crack),
    ('Delamination Crack', delamination_crack),
]

for i, (name, gen_fn) in enumerate(generators):
    rng = np.random.default_rng(42 + i)
    mask, meta = gen_fn(256, 256, rng=rng)
    
    # Show mask
    axes[0, i].imshow(mask, cmap='gray')
    axes[0, i].set_title(f'{name}\nMask')
    axes[0, i].axis('off')
    
    # Render on USM background
    bg = np.full((256, 256), 0.4, dtype=np.float32)
    usm = render_crack_on_usm(bg, mask, rng=rng)
    axes[1, i].imshow(usm, cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title('On USM')
    axes[1, i].axis('off')

plt.suptitle('Semiconductor Crack Types', fontsize=14)
plt.tight_layout()
plt.show()

## Full Synthetic USM Generation

In [ ]:
rng = np.random.default_rng(99)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    img, mask, meta = generate_synthetic_usm_with_cracks(
        height=256, width=256, n_cracks=1 + i, rng=rng,
    )
    axes[0, i].imshow(img, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'{meta["n_cracks"]} crack(s)')
    axes[0, i].axis('off')
    axes[1, i].imshow(mask, cmap='gray')
    axes[1, i].set_title('Mask')
    axes[1, i].axis('off')

plt.suptitle('Synthetic USM Images with Cracks', fontsize=14)
plt.tight_layout()
plt.show()